In [ ]:
!pip install langchain langchain-community langchain-core
!pip install sentence-transformers faiss-cpu
!pip install huggingface-hub
!pip install gradio

In [ ]:
! pip install langchain-openai google-generativeai
! pip install --upgrade langchain langchain-community



In [9]:
! pip install -q gradio


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_community.document_loaders import PyPDFLoader, UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOllama

In [10]:
# Embedding using BGE-large-en
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")
persist_path = "./faiss_index"
req_file_name = ""

In [1]:
print("Hello World!")

Hello World!


In [13]:
!pip install pypdf

  Using cached pypdf-5.8.0-py3-none-any.whl.metadata (7.1 kB)
Using cached pypdf-5.8.0-py3-none-any.whl (309 kB)



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import gradio as gr
import os
import shutil
import textwrap
import nest_asyncio
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOllama

nest_asyncio.apply()
load_dotenv(find_dotenv())

# Global variables
persist_path = "./faiss_index"
req_file_name = ""

# Load embedding model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")


def wrap_text_preserve_newlines(text, width=110):
    lines = text.split('\n')
    return '\n'.join([textwrap.fill(line, width=width) for line in lines])


def process_llm_response(llm_response):
    if 'result' not in llm_response or 'source_documents' not in llm_response:
        return "Invalid LLM response format."

    response = wrap_text_preserve_newlines(llm_response['result'])
    sources = "\n\nSources:\n" + "\n".join(
        source.metadata.get('source', 'Unknown') for source in llm_response["source_documents"]
    )
    return response + sources


def save_response_to_file(response, filename):
    with open(filename, 'w') as f:
        f.write(response + '\n')


def save_files_req(uploaded_file):
    global req_file_name
    os.makedirs("./requirements", exist_ok=True)
    file_name = os.path.basename(uploaded_file.name)
    req_file_name = file_name
    file_path = os.path.join("./requirements", file_name)
    shutil.copy(uploaded_file.name, file_path)
    return f"✅ Requirement File Saved: {file_name}"


def save_files_policies(uploaded_files):
    os.makedirs("./policies", exist_ok=True)
    saved = []
    for file in uploaded_files:
        name = os.path.basename(file.name)
        path = os.path.join("./policies", name)
        shutil.copy(file.name, path)
        saved.append(name)
    return f"✅ Policies Saved: {', '.join(saved)}"


def process_doc():
    global req_file_name
    loader = PyPDFLoader(f"./requirements/{req_file_name}")
    pages = loader.load_and_split()
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=1000, chunk_overlap=100)
    docs = splitter.split_documents(pages)

    db = FAISS.from_documents(documents=docs, embedding=embeddings)
    db.save_local(persist_path)
    return "✅ FAISS vectorstore created."


def talk_to_model(selected_model):
    db = FAISS.load_local(persist_path, embeddings, allow_dangerous_deserialization=True)
    retriever = db.as_retriever(search_kwargs={"k": 5})

    try:
        with open("prompt.txt", "r") as f:
            question = f.read()
    except FileNotFoundError:
        return "❌ prompt.txt not found."

    prompt = PromptTemplate(
        input_variables=["history", "context", "question"],
        template="""
You are a software compliance checker for GDPR.
Use the context below between <ctx></ctx> and chat history <hs></hs> to answer the question:
<ctx>
{context}
</ctx>
<hs>
{history}
</hs>
Question: {question}
Answer:
""")

    llm = ChatOllama(model=selected_model, temperature=0.3)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={
            "verbose": False,
            "prompt": prompt,
            "memory": ConversationBufferMemory(memory_key="history", input_key="question")
        }
    )

    response = qa_chain(question)
    result = process_llm_response(response)
    save_response_to_file(result, "./ollama_report.txt")
    return result


# ----------------- Gradio UI -----------------

with gr.Blocks() as demo:
    gr.Markdown("## GDPR Compliance Checker using Ollama + BGE + FAISS")

    with gr.Row():
        uploaded_files_policies = gr.File(file_count="multiple", label="Upload Policy PDFs")
        uploaded_files_req = gr.File(file_count="single", file_types=[".pdf"], label="Upload Requirements PDF")

    with gr.Row():
        save_btn_policies = gr.Button("Save Policy Files")
        save_btn_req = gr.Button("Save Requirement File")
        process_btn = gr.Button("Step 3: Process Documents")
        run_btn = gr.Button("Step 4: Run Compliance Check")

    model_choice = gr.Dropdown(["qwq:32b", "gemma:27b"], label="Select Ollama Model")

    status = gr.Textbox(label="Status", interactive=False)
    output = gr.TextArea(label="Compliance Report", lines=20)

    save_btn_policies.click(save_files_policies, inputs=[uploaded_files_policies], outputs=[status])
    save_btn_req.click(save_files_req, inputs=[uploaded_files_req], outputs=[status])
    process_btn.click(process_doc, outputs=[status])
    run_btn.click(talk_to_model, inputs=[model_choice], outputs=[output])

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


C:\Users\souvick.das\AppData\Local\Temp\ipykernel_39508\1152424847.py:106: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=selected_model, temperature=0.3)
C:\Users\souvick.das\AppData\Local\Temp\ipykernel_39508\1152424847.py:116: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  "memory": ConversationBufferMemory(memory_key="history", input_key="question")
C:\Users\souvick.das\AppData\Local\Temp\ipykernel_39508\1152424847.py:120: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa

In [15]:
import os
import shutil
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Paths
SRS_DIR = "./requirements"
GDPR_DIR = "./policies"
SRS_INDEX_PATH = "./SRS_index"
GDPR_INDEX_PATH = "./GDPR_index"

# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")

# Document splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

def create_vectorstore_from_dir(directory, index_path):
    documents = []
    for file in os.listdir(directory):
        if file.endswith(".pdf"):
            path = os.path.join(directory, file)
            loader = PyPDFLoader(path)
            docs = loader.load_and_split()
            documents.extend(docs)

    print(f"Loaded {len(documents)} documents from {directory}")
    split_docs = splitter.split_documents(documents)
    print(f"Split into {len(split_docs)} chunks")

    vectorstore = FAISS.from_documents(split_docs, embedding=embeddings)
    vectorstore.save_local(index_path)
    print(f"Saved FAISS vectorstore to: {index_path}\n")

if __name__ == "__main__":
    os.makedirs(SRS_INDEX_PATH, exist_ok=True)
    os.makedirs(GDPR_INDEX_PATH, exist_ok=True)

    print("\n>>> Building SRS vectorstore...")
    create_vectorstore_from_dir(SRS_DIR, SRS_INDEX_PATH)

    print("\n>>> Building GDPR vectorstore...")
    create_vectorstore_from_dir(GDPR_DIR, GDPR_INDEX_PATH)



>>> Building SRS vectorstore...
Loaded 5 documents from ./requirements
Split into 9 chunks
Saved FAISS vectorstore to: ./SRS_index


>>> Building GDPR vectorstore...
Loaded 136 documents from ./policies
Split into 477 chunks
Saved FAISS vectorstore to: ./GDPR_index



In [17]:
!pip install llama-parse

  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached Deprecated-1.2.18-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached wrapt-1.17.2-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached griffe-1.7.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/47.9 kB ? eta -:--:--
   ---------------------------------------- 47.9/47.9 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/289.1 kB ? eta -:--:--
   ---------------------------------------- 289.1/289.1 kB 9.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.3 MB 24.1 MB/s eta 0:00:01
   ------ --------------------------------- 1.8/10.3 MB 22.4 MB/s eta 0:00:01
   ---


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
from llama_parse import LlamaParse
from dotenv import load_dotenv, find_dotenv
import nest_asyncio


load_dotenv(find_dotenv())

# Paths
REQUIREMENTS_PDF = "./requirements/e-commerce.pdf"  # replace with actual file name
PROMPT_PATH = "prompt.txt"
LLAMA_INDEX_KEY = os.environ.get("LLAMA_PARSE_API_KEY")

nest_asyncio.apply()


# llx-mENzHsPtyYIGJX9ZyNisiLrm68oB4kYd18764Klz21YYILOA



# Initialize parser
parser = LlamaParse(
    api_key=LLAMA_INDEX_KEY,  # can also be set in your env as LLAMA_CLOUD_API_KEY
    result_type="markdown",  # "markdown" and "text" are available
    num_workers=4,  # if multiple files passed, split in `num_workers` API calls
    verbose=True,
    language="en",  # Optionally you can define a language, default=en
)

# Load and convert PDF to markdown
documents = parser.load_data([REQUIREMENTS_PDF])
merged_text = "\n".join([doc.text for doc in documents])

# Read prompt content
with open(PROMPT_PATH, "r") as f:
    prompt_instruction = f.read()

# Prepend prompt content to the parsed requirements
combined_input = prompt_instruction + "\n\n" + merged_text

# Now `combined_input` can be passed to the CC_Agent as query
print("\n>>> Combined prompt and requirements text prepared.")

# Optional: save to file
with open("combined_prompt_input.txt", "w", encoding="utf-8") as f:
    f.write(combined_input)

print("\n>>> Saved combined input to combined_prompt_input.txt")


Parsing files:   0%|          | 0/1 [00:00<?, ?it/s]

Started parsing the file under job_id f1719cad-8b8c-47b7-83e2-0933a5a8368f


Parsing files: 100%|██████████| 1/1 [00:24<00:00, 24.16s/it]


>>> Combined prompt and requirements text prepared.

>>> Saved combined input to combined_prompt_input.txt


In [ ]:
import os
import shutil
import textwrap
import nest_asyncio
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOllama

nest_asyncio.apply()
load_dotenv(find_dotenv())

# Paths
SRS_PATH = "./requirements"
SRS_INDEX_PATH = "./SRS_index"
GDPR_INDEX_PATH = "./GDPR_index"

# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")

# Load vector stores
srs_vectorstore = FAISS.load_local(SRS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
gdpr_vectorstore = FAISS.load_local(GDPR_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)

# Prompts
cc_prompt = PromptTemplate(
    input_variables=["history", "context", "question"],
    template="""
You are CC_Agent, a GDPR compliance checker.
Your task is to analyze the provided software requirements (SRS) for GDPR compliance.
Use the context below to:
- Identify compliance issues
- Highlight strengths/weaknesses
- Suggest actionable improvements

<ctx>
{context}
</ctx>
<hs>
{history}
</hs>
Question:
{question}
Answer:
"""
)

ra_prompt = PromptTemplate(
    input_variables=["history", "context", "question"],
    template="""
You are RA_Agent, a GDPR legal reviewer.
You received a GDPR compliance report from another agent (CC_Agent).
Your responsibilities:
- Validate the report using official GDPR context
- Cite articles/sections that support or challenge points
- Instruct CC_Agent for reevaluation if needed
- Produce a final report if everything is valid

<ctx>
{context}
</ctx>
<hs>
{history}
</hs>
Compliance Report:
{question}
Answer:
"""
)

# Initialize agents
cc_llm = ChatOllama(model="gemma3:27b", temperature=0.3)
ra_llm = ChatOllama(model="qwq:32b", temperature=0.3)

cc_chain = RetrievalQA.from_chain_type(
    llm=cc_llm,
    retriever=srs_vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": cc_prompt,
        "memory": ConversationBufferMemory(memory_key="history", input_key="question")
    }
)

ra_chain = RetrievalQA.from_chain_type(
    llm=ra_llm,
    retriever=gdpr_vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": ra_prompt,
        "memory": ConversationBufferMemory(memory_key="history", input_key="question")
    }
)

# Load SRS query prompt
with open("combined_prompt_input.txt", "r") as f:
    srs_query = f.read()

# === CC Agent Initial Review ===
cc_output1 = cc_chain(srs_query)
cc_report1 = cc_output1['result']
print("\n===== CC Agent Initial Output =====\n")
print(cc_report1)

# === RA Agent Review ===
ra_feedback1 = ra_chain(cc_report1)
ra_review = ra_feedback1['result']
print("\n===== RA Agent Feedback =====\n")
print(ra_review)

# === Second Review Loop ===
if "re-evaluate" in ra_review.lower() or "reassess" in ra_review.lower():
    print("\nRA Agent requested reassessment.\n")
    cc_output2 = cc_chain(ra_review)
    cc_report2 = cc_output2['result']
    print("\n===== CC Agent Second Assessment =====\n")
    print(cc_report2)

    ra_final = ra_chain(cc_report2)
    final_report = ra_final['result']
else:
    final_report = ra_review

# === Final Report Output ===
print("\n===== Final Comprehensive Report =====\n")
print(final_report)

with open("final_compliance_report.txt", "w") as f:
    f.write(final_report)



===== CC Agent Initial Output =====

## GDPR Compliance Review: E-commerce Application SRS

Here's a GDPR compliance review of the provided Software Requirements Specification (SRS) excerpts.

**1. Analysis & Identification of Non-Compliance/Potential Issues:**

* **2.1 User Registration:** While email verification is good practice, the SRS lacks explicit mention of obtaining *explicit consent* for data processing during registration. Simply collecting data isn't sufficient; users must actively consent to how their data will be used (GDPR Article 6).
* **2.1 User Registration & 3.3 User Profile Management:** The SRS details collection of “necessary details” and allows updating “personal information” but doesn’t specify *what* data is collected, *why* it’s collected, or how long it’s retained. This violates the data minimization principle (GDPR Article 5(1)(c)) and the right to be informed (GDPR Article 13).
* **2.1 User Authentication (Social Login):**  If social login is enabled, the

In [29]:
import os
import nest_asyncio
from dotenv import load_dotenv, find_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_models import ChatOllama
# CC Agent (uses full document input, no retrieval)
from langchain.chains import LLMChain



nest_asyncio.apply()
load_dotenv(find_dotenv())

# Paths
GDPR_INDEX_PATH = "./GDPR_index"
COMBINED_INPUT_PATH = "combined_prompt_input.txt"

# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")

# Load only the GDPR vector store (SRS is provided as plain input)
gdpr_vectorstore = FAISS.load_local(GDPR_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)

# Prompts
cc_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are CC_Agent, a GDPR compliance checker.
You are given the full software requirements specification (SRS) as part of the question below.

Analyze it for GDPR compliance and:
- Identify compliance issues
- Highlight strengths and weaknesses
- Suggest actionable improvements

SRS + Prompt:
{question}
"""
)

# ra_prompt = PromptTemplate(
#     input_variables=["history", "context", "question"],
#     template="""
# You are RA_Agent, a GDPR legal reviewer.
# You received a GDPR compliance report from another agent (CC_Agent).
# Your responsibilities:
# - Validate the report using official GDPR context
# - Cite articles/sections that support or challenge points
# - Instruct CC_Agent for reevaluation if needed
# - Produce a final report if everything is valid

# <ctx>
# {context}
# </ctx>
# <hs>
# {history}
# </hs>
# Compliance Report:
# {question}
# Answer:
# """
# )

ra_prompt = PromptTemplate(
    input_variables=["history", "context", "question"],
    template="""
You are RA_Agent, a GDPR legal reviewer.
You received a GDPR compliance report from another agent (CC_Agent).

Your responsibilities:
- Validate the report using official GDPR context
- Cite articles/sections that support or challenge points
- Decide if CC_Agent must reassess the document

Respond with a JSON in the format:
{{
  "re_evaluate_required": true/false,
  "justification": "...",
  "final_review": "..."
}}

<ctx>
{context}
</ctx>
<hs>
{history}
</hs>
Compliance Report:
{question}
Answer:
"""
)


# Initialize agents
cc_llm = ChatOllama(model="gemma3:27b", temperature=0.3)
ra_llm = ChatOllama(model="qwq:32b", temperature=0.3)




# CC Agent (uses full document input, no retrieval)

cc_chain = LLMChain(llm=cc_llm, prompt=cc_prompt)

# RA Agent (uses GDPR vector store)
ra_chain = RetrievalQA.from_chain_type(
    llm=ra_llm,
    retriever=gdpr_vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": ra_prompt,
        "memory": ConversationBufferMemory(memory_key="history", input_key="question")
    }
)

# Load SRS query prompt (entire SRS + prompt already combined)
with open(COMBINED_INPUT_PATH, "r", encoding="utf-8") as f:
    srs_query = f.read()

# === CC Agent Initial Review ===
cc_output1 = cc_chain.run(srs_query)
print("\n===== CC Agent Initial Output =====\n")
print(cc_output1)





# === RA Agent Review ===
# ra_feedback1 = ra_chain.run(cc_output1)
# print("\n===== RA Agent Feedback =====\n")
# print(ra_feedback1)

# === Second Review Loop ===
# if "re-evaluate" in ra_feedback1.lower() or "reassess" in ra_feedback1.lower():
#     print("\nRA Agent requested reassessment.\n")
#     cc_output2 = cc_chain.run(ra_feedback1)
#     print("\n===== CC Agent Second Assessment =====\n")
#     print(cc_output2)

#     ra_final = ra_chain.run(cc_output2)
#     final_report = ra_final
# else:
#     final_report = ra_feedback1

# # === Final Report Output ===
# print("\n===== Final Comprehensive Report =====\n")
# print(final_report)

# with open("final_compliance_report.txt", "w", encoding="utf-8") as f:
#     f.write(final_report)



===== CC Agent Initial Output =====

## GDPR Compliance Review - E-commerce Application SRS

Here's a GDPR compliance review of the provided Software Requirements Specification (SRS) excerpts.

**Overall Assessment:** The SRS demonstrates *some* awareness of GDPR, particularly in section 5.3 (Security) and 5 (Legal and Compliance Requirements). However, it lacks detailed specifications on *how* GDPR principles will be implemented throughout the application.  The document primarily *states* compliance needs rather than detailing the *mechanisms* to achieve it.

**1. Compliance Issues & Recommendations:**

* **2.1 User Management – Lack of Explicit Consent Mechanisms:** While 5.1 mentions obtaining user consent, the User Registration (2.1.1) requirement doesn’t specify *how* consent for data processing (beyond account creation) will be obtained.  
    * **Recommendation:** Add a requirement specifying granular consent options during registration and within user profile settings.  Users 

In [24]:
# === RA Agent Review ===
ra_feedback1 = ra_chain(cc_output1)  # or: ra_chain.invoke(cc_output1)
ra_review = ra_feedback1['result']

print("\n===== RA Agent Feedback =====\n")
print(ra_review)



===== RA Agent Feedback =====

<think>
Okay, let me start by reviewing the provided GDPR compliance report from CC_Agent and the context they included. My role is to validate their findings using official GDPR articles and sections, cite relevant articles, and determine if any reevaluation is needed.

First, looking at the Compliance Issues and Recommendations section. The report mentions several areas like explicit consent for newsletters, user rights (access, rectification, erasure, portability), guest checkout concerns, data deletion processes, encryption specifics, and consent management. 

Starting with the first point: explicit consent for newsletters. The report recommends affirmative opt-in. Under GDPR, Article 7 requires consent to be freely given, specific, informed, and unambiguous. An opt-in checkbox that's separate from other terms aligns with this. So that's correct. The report's recommendation is valid here.

Next, the user rights (Articles 15-20). The SRS doesn't menti

In [28]:
# === Second Review Loop ===
if "re-evaluate" in ra_review.lower() or "reassess" in ra_review.lower():
    print("\nRA Agent requested reassessment.\n")
    cc_output2 = cc_chain.run(ra_feedback1['result'])
    print("\n===== CC Agent Second Assessment =====\n")
    print(cc_output2)

    ra_final = ra_chain(cc_output2)
    final_report = ra_final['result']
else:
    final_report = ra_feedback1['result']

# === Final Report Output ===
print("\n===== Final Comprehensive Report =====\n")
print(final_report)

with open("final_compliance_report.txt", "w", encoding="utf-8") as f:
    f.write(final_report)


===== Final Comprehensive Report =====

<think>
Okay, let me start by reviewing the provided GDPR compliance report from CC_Agent and the context they included. My role is to validate their findings using official GDPR articles and sections, cite relevant articles, and determine if any reevaluation is needed.

First, looking at the Compliance Issues and Recommendations section. The report mentions several areas like explicit consent for newsletters, user rights (access, rectification, erasure, portability), guest checkout concerns, data deletion processes, encryption specifics, and consent management. 

Starting with the first point: explicit consent for newsletters. The report recommends affirmative opt-in. Under GDPR, Article 7 requires consent to be freely given, specific, informed, and unambiguous. An opt-in checkbox that's separate from other terms aligns with this. So that's correct. The report's recommendation is valid here.

Next, the user rights (Articles 15-20). The SRS does

In [33]:
import json
import re

# RA Agent output (possibly contains <think>...</think> followed by JSON)
ra_feedback1 = ra_chain(cc_output1)
raw_result = ra_feedback1['result']

# Step 1: Strip out the <think>...</think> block using regex
json_part = re.sub(r"<think>.*?</think>", "", raw_result, flags=re.DOTALL).strip()

# Step 2: Extract JSON block from response
try:
    ra_response = json.loads(json_part)
    ra_review = ra_response["final_review"]

    if ra_response["re_evaluate_required"]:
        print("\nRA Agent requested reassessment.\n")
        cc_output2 = cc_chain.run(ra_response["justification"])
        print("\n===== CC Agent Second Assessment =====\n")
        print(cc_output2)

        ra_final = ra_chain(cc_output2)
        final_json = json.loads(re.sub(r"<think>.*?</think>", "", ra_final['result'], flags=re.DOTALL).strip())
        final_report = final_json["final_review"]
    else:
        final_report = ra_review

except json.JSONDecodeError as e:
    print("⚠️ Failed to parse RA Agent's response. Raw result below:\n")
    print(raw_result)
    final_report = raw_result


⚠️ Failed to parse RA Agent's response. Raw result below:

<think>
Okay, let me tackle this query. The user provided a detailed GDPR compliance report for an e-commerce application's SRS and wants me to evaluate whether the report is accurate and if a re-evaluation is needed. 

First, I need to recall the key GDPR articles and principles. The report mentions several areas like consent, data minimization, data subject rights, breach notifications, DPAs, and DPIAs. Let me cross-check each of these against the GDPR requirements.

Starting with consent (Article 7). The report says the SRS doesn't specify how consent is obtained beyond account creation. GDPR requires consent to be explicit, so the recommendation for granular consent options makes sense. That's correct.

Data minimization (Article 5(1)(c)) is another point. The SRS allows profile pictures but doesn't state why or how long they're kept. The report's recommendation to define retention periods and necessity aligns with GDPR's d

In [39]:
print(final_report)

<think>
Okay, let me tackle this query. The user provided a detailed GDPR compliance report for an e-commerce application's SRS and wants me to evaluate whether the report is accurate and if a re-evaluation is needed. 

First, I need to recall the key GDPR articles and principles. The report mentions several areas like consent, data minimization, data subject rights, breach notifications, DPAs, and DPIAs. Let me cross-check each of these against the GDPR requirements.

Starting with consent (Article 7). The report says the SRS doesn't specify how consent is obtained beyond account creation. GDPR requires consent to be explicit, so the recommendation for granular consent options makes sense. That's correct.

Data minimization (Article 5(1)(c)) is another point. The SRS allows profile pictures but doesn't state why or how long they're kept. The report's recommendation to define retention periods and necessity aligns with GDPR's data minimization principle. Good.

Data subject rights (Art

In [ ]:
import json
import re

def extract_re_evaluation_flag(raw_text: str) -> bool:
    try:
        # Strip out <think>...</think>
        cleaned_text = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()

        # Find JSON block manually using curly brace pattern
        match = re.search(r"\{.*?\}", cleaned_text, flags=re.DOTALL)
        if match:
            json_str = match.group(0)
            data = json.loads(json_str)
            # print("Extracted JSON:", data.get("re_evaluate_required", True))

            return data.get("re_evaluate_required", False)
        else:
            print("⚠️ No JSON block found.")
            return False

    except json.JSONDecodeError as e:
        print("⚠️ JSON decoding failed:", e)
        return False


In [38]:
raw_result = ra_feedback1["result"]

needs_reassessment = extract_re_evaluation_flag(raw_result)

if needs_reassessment:
    print("🔁 RA_Agent requires a second evaluation.")
else:
    print("✅ RA_Agent considers the report final.")

Extracted JSON: False
✅ RA_Agent considers the report final.


HITL

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_community.chat_models import ChatOllama

# Load the documents
with open("reports/ra_initial_report.txt", "r") as f:
    original_report = f.read()

with open("reports/human_edited_report.txt", "r") as f:
    edited_report = f.read()

with open("combined_prompt_input.txt", "r") as f:
    srs_full = f.read()

# MemAgent Prompt
mem_prompt = PromptTemplate(
    input_variables=["original", "edited", "srs"],
    template="""
You are MemAgent, a compliance traceability and audit assistant.

Your job is to compare two versions of a GDPR compliance report:
1. The original generated by RA_Agent
2. The version edited by a human expert

Also refer to the original software requirements specification (SRS).

Identify:
- Key differences in content
- Which sections were modified, removed, or added
- The likely rationale for the human changes
- Whether the changes align better with the original requirements

Format the output as a clear comparison summary.

--- Original Report ---
{original}

--- Human Edited Report ---
{edited}

--- SRS Document ---
{srs}

Summary:
"""
)

# Run the MemAgent
mem_llm = ChatOllama(model="gemma3:27b", temperature=0.3)
mem_chain = LLMChain(llm=mem_llm, prompt=mem_prompt)

mem_summary = mem_chain.run({
    "original": original_report,
    "edited": edited_report,
    "srs": srs_full
})

# Save summary
with open("reports/memagent_change_summary.txt", "w") as f:
    f.write(mem_summary)

print("\n✅ MemAgent summary saved to reports/memagent_change_summary.txt")
